In [ ]:
%use kandy
%use lets-plot
%use dataframe

## Header Analysis

In [ ]:
import java.io.File
import org.jetbrains.kotlinx.dataframe.DataFrame
import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.api.toStart

fun groupCsvFilesByHeader(directoryPath: String): Map<String, List<File>> {
  val directory = File(directoryPath)

  if (!directory.isDirectory) {
    throw IllegalArgumentException("The provided path '$directoryPath' is not a valid directory.")
  }

  return directory
      .walkTopDown()
      .filter { it.isFile && it.extension.equals("csv", ignoreCase = true) }
      .groupBy { file ->
        try {

          //
          // println("""\d{2,4}-\d{2}""".toRegex().find(file.name.substringBeforeLast(".csv"))?.value)
          //          println("Reading ${file.absolutePath}...")
          //
          //
          // """\d{2,4}-\d{2}""".toRegex().find(file.name.substringBeforeLast(".csv"))?.value?.let {
          //              year ->
          //            addYearColumn(year, file.absolutePath, file.absolutePath)
          //          }

          //				splitDiagnosisCode_Description(file.absolutePath, file.absolutePath)
          //				removeEmptyColumnsWithDataFrame(file.absolutePath, file.absolutePath)
          //				println("Successfully removed empty columns using DataFrame!")

          //			removeTextFromHeader(file.absolutePath, file.absolutePath, "Table \\d+\\. ")

          //                removeFirstColumnWithDataFrame(file.absolutePath, file.absolutePath)
          //				println("Successfully removed first column using DataFrame!")

        } catch (e: Exception) {
          println("Error processing dataframe: ${e.message}")
        }

        // useLines automatically closes the file after reading what we need
        file.useLines { lines -> lines.firstOrNull() ?: "<EMPTY FILE>" }
      }
}

fun walkDirectory(directoryPath: String) {
  val directory = File(directoryPath)

  if (!directory.isDirectory) {
    throw IllegalArgumentException("The provided path '$directoryPath' is not a valid directory.")
  }

  val dataframes = mutableMapOf<String, MutableList<DataFrame<*>>>()

  directory
      .walkTopDown()
      .filter { it.isFile && it.extension.equals("csv", ignoreCase = true) }
      .forEach { file ->
        try {
          createDFList(file.absolutePath).let { (type, df) ->
            dataframes.getOrPut(type, { mutableListOf() }).add(df)
          }

          //
          // println("""\d{2,4}-\d{2}""".toRegex().find(file.name.substringBeforeLast(".csv"))?.value)
          //          println("Reading ${file.absolutePath}...")
          //
          //
          // """\d{2,4}-\d{2}""".toRegex().find(file.name.substringBeforeLast(".csv"))?.value?.let {
          //              year ->
          //            addYearColumn(year, file.absolutePath, file.absolutePath)
          //          }

          //				splitDiagnosisCode_Description(file.absolutePath, file.absolutePath)
          //				removeEmptyColumnsWithDataFrame(file.absolutePath, file.absolutePath)
          //				println("Successfully removed empty columns using DataFrame!")

          //			removeTextFromHeader(file.absolutePath, file.absolutePath, "Table \\d+\\. ")

          //                removeFirstColumnWithDataFrame(file.absolutePath, file.absolutePath)
          //				println("Successfully removed first column using DataFrame!")

        } catch (e: Exception) {
          println("Error processing dataframe: ${e.message}")
        }
      }

  dataframes.forEach { (type, dfs) ->
    println("$type: ${dfs.count()}")
    var mergedDF = dfs[0]
    dfs.subList(1, dfs.lastIndex).forEach { df -> mergedDF = mergedDF.concat(df) }

    mergedDF.writeCsv(
        directory.absolutePath.plus("/${type.replace(":", " –")}_v2.csv"),
        includeHeader = true,
        quote = '"',
    )
  }
}

fun createDFList(inputPath: String): Pair<String, DataFrame<*>> {
  val df = DataFrame.readCsv(inputPath)

  return Pair(df.columnNames().get(1), df)
}

fun addYearColumn(year: String, inputPath: String, outputPath: String) {
  val df = DataFrame.readCsv(inputPath)
  val cleanedDf = df.insert("Year") { year }.at(0)

  cleanedDf.writeCsv(
      outputPath,
      includeHeader = true,
      quote = '"',
  )
}

fun splitDiagnosisCode_Description(inputPath: String, outputPath: String) {
  val df = DataFrame.readCsv(inputPath)

  val colName = "Primary Diagnosis: 4 character code and description"

  val newName = "Primary Diagnosis: 4 character code"

  if (!df.containsColumn(colName) || df.containsColumn("Description")) {
    return
  }

  val cleanedDf =
      df.add(newName) { (it[colName] as? String)?.split(" ", limit = 2)?.getOrNull(0) }
          .add("Description") { (it[colName] as? String)?.split(" ", limit = 2)?.getOrNull(1) }
          .remove(colName)
          .move(newName, "Description")
          .toStart()

  cleanedDf.writeCsv(
      outputPath,
      includeHeader = true,
      quote = '"',
  )
}

fun removeEmptyColumnsWithDataFrame(inputPath: String, outputPath: String) {
  val df = DataFrame.readCsv(inputPath)

  val cleanedDf =
      df.remove {
        cols { col ->
          col.values().all { value -> value == null || value.toString().isBlank() }
        }
      }

  cleanedDf.writeCsv(
      outputPath,
      includeHeader = true,
      quote = '"',
  )
}

fun removeFirstColumnWithDataFrame(inputPath: String, outputPath: String) {
  val df = DataFrame.readCsv(inputPath)

  if (df.columnsCount() > 0) {
    val firstColumnName = df.columnNames().first()

    val cleanedDf = df.remove(firstColumnName)

    cleanedDf.writeCsv(outputPath)
  } else {
    println("The CSV is completely empty.")
  }
}

fun removeTextFromHeader(inputPath: String, outputPath: String, textToRemove: String) {

  val df = DataFrame.readCsv(inputPath)

  val cleanedDf =
      df.rename { all() }.into { originalName -> originalName.name.replace(textToRemove, "") }

  cleanedDf.writeCsv(outputPath)
}

fun main() {
  val targetDirectory =
      "./COMP4037 - Research Methods/Coursework 2/NHS Hospital Admissions 2"

  try {

//    walkDirectory(targetDirectory)

    //    val groupedFiles = groupCsvFilesByHeader(targetDirectory)
    //
    //    groupedFiles.forEach { (header, files) ->
    //      println("Header: \n$header")
    //      files.forEach { file -> println("  - ${file.absolutePath}") }
    //      println("---")
    //    }
  } catch (e: Exception) {
    println("Error: ${e.message}")
  }
}

main()

In [ ]:
val targetDirectory =
	"./COMP4037 - Research Methods/Coursework 2/NHS Hospital Admissions 2"

val allDiagnosis4 = DataFrame.readCsv(targetDirectory.plus("/").plus("All diagnoses – 4 character code and description_v2.csv"))
//val allDiagnosis3 = DataFrame.readCsv(targetDirectory.plus("/").plus("All diagnoses – 3 character code and description_v2.csv"))
//val primaryDiagnosis3 = DataFrame.readCsv(targetDirectory.plus("/").plus("Primary Diagnosis – 3 character code and description_v2.csv"))
//val primaryDiagnosis4 = DataFrame.readCsv(targetDirectory.plus("/").plus("Primary Diagnosis – 4 character code and description_v2.csv"))
//val primaryDiagnosisSummary = DataFrame.readCsv(targetDirectory.plus("/").plus("Primary Diagnosis – summary code and description_v2.csv"))

In [ ]:
primaryDiagnosisSummary.columnNames()

In [ ]:
primaryDiagnosisSummary.plot {
	line {
		x(Year)
		y(`Median time waited`)
		color(Description)
	}
}

In [ ]:
primaryDiagnosis3.describe()

In [ ]:
primaryDiagnosis4.describe()

In [ ]:
primaryDiagnosisSummary.describe()

In [ ]:
primaryDiagnosis3
	.groupBy { `Primary Diagnosis - 3 character code and description` }.schema()

In [ ]:
allDiagnosis4.columnNames()

In [ ]:
val fullyCleanDf = allDiagnosis4.convert {
	colsOf<String?>().except("NHS Period", "All diagnoses: 4 character code and description", "Description")
}.with { value ->
	val str = value?.toString()?.trim()
	if (str == "*" || str.isNullOrEmpty()) 0.0 else str.toDoubleOrNull() ?: 0.0
}

In [ ]:
fullyCleanDf

In [ ]:
val dfWithPercentChange = fullyCleanDf
	.groupBy { `Primary Diagnosis - 3 character code and description` }
	.updateGroups { groupDf ->
		groupDf.sortBy { column<Int>("Year") }
			.add("AdmissionsPercentChange") {
				val current = "Admissions"<Double>()

				val previous = prev()?.get("Admissions") as? Double

				// 5. Apply the math formula
				if (previous != null && previous != 0.0) {
					((current - previous) / previous) * 100.0
				} else {
					null
				}
			}
	}

In [ ]:
dfWithPercentChange

In [ ]:
dfWithPercentChange.groups.forEach {
	val code = it.`Primary Diagnosis – 3 character code and description`.distinct()
	it.writeCsv(targetDirectory.plus("/").plus("Primary Diagnosis – 3 character code and description_${code}_cleaned.csv"))
}



In [ ]:
dfWithPercentChange.groups

	. { index, df ->
	println(index)
	if (index > 0) {
		mergedDF = mergedDF.concat(df)
	}
}

## ONS Population Data & NHS Primary Diagnosis – 3 Character Code Cleaned

In [ ]:
//    Chapter 	Block 	    Title
//    I        	A00–B99 	Certain infectious and parasitic diseases
//    II 	    C00–D48 	Neoplasms
//    III 	    D50–D89 	Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism
//    IV 	    E00–E90 	Endocrine, nutritional and metabolic diseases
//    V 	    F00–F99 	Mental and behavioural disorders
//    VI 	    G00–G99 	Diseases of the nervous system
//    VII     	H00–H59 	Diseases of the eye and adnexa
//    VIII 	    H60–H95 	Diseases of the ear and mastoid process
//    IX      	I00–I99 	Diseases of the circulatory system
//    X 	    J00–J99 	Diseases of the respiratory system
//    XI 	    K00–K93 	Diseases of the digestive system
//    XII 	    L00–L99 	Diseases of the skin and subcutaneous tissue
//    XIII 	    M00–M99 	Diseases of the musculoskeletal system and connective tissue
//    XIV 	    N00–N99 	Diseases of the genitourinary system
//    XV 	    O00–O99 	Pregnancy, childbirth and the puerperium
//    XVI 	    P00–P96 	Certain conditions originating in the perinatal period
//    XVII 	    Q00–Q99 	Congenital malformations, deformations and chromosomal abnormalities
//    XVIII 	R00–R99 	Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified
//    XIX 	    S00–T98 	Injury, poisoning and certain other consequences of external causes
//    XX 	    V01–Y98 	External causes of morbidity and mortality
//    XXI 	    Z00–Z99 	Factors influencing health status and contact with health services
//    XXII 	    U00–U99 	Codes for special purposes

In [ ]:
fun mapIcd10Chapter(code: String?): String {
	if (code.isNullOrBlank() || code.length < 3) return "Unknown or Invalid"

	val cleanCode = code.trim().uppercase()
	val letter = cleanCode[0]
	val number = cleanCode.substring(1, 3).toIntOrNull() ?: return "Unknown or Invalid"

	return when (letter) {
		'A', 'B' -> "Certain infectious and parasitic diseases" // I
		'C' -> "Neoplasms" // II
		'D' -> when (number) {
			in 0..48 -> "Neoplasms" // II
			in 50..89 -> "Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism" // III
			else -> "Unknown or Invalid"
		}
		'E' -> "Endocrine, nutritional and metabolic diseases" // IV
		'F' -> "Mental and behavioural disorders" // V
		'G' -> "Diseases of the nervous system" // VI
		'H' -> when (number) {
			in 0..59 -> "Diseases of the eye and adnexa" // VII
			in 60..95 -> "Diseases of the ear and mastoid process" // VIII
			else -> "Unknown or Invalid"
		}
		'I' -> "Diseases of the circulatory system" // IX
		'J' -> "Diseases of the respiratory system" // X
		'K' -> "Diseases of the digestive system" // XI
		'L' -> "Diseases of the skin and subcutaneous tissue" // XII
		'M' -> "Diseases of the musculoskeletal system and connective tissue" // XIII
		'N' -> "Diseases of the genitourinary system" // XIV
		'O' -> "Pregnancy, childbirth and the puerperium" // XV
		'P' -> "Certain conditions originating in the perinatal period" // XVI
		'Q' -> "Congenital malformations, deformations and chromosomal abnormalities" // XVII
		'R' -> "Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified" // XVIII
		'S', 'T' -> "Injury, poisoning and certain other consequences of external causes" // XIX
		'V', 'W', 'X', 'Y' -> "External causes of morbidity and mortality" //  XX
		'Z' -> "Factors influencing health status and contact with health services" // XXI
		'U' -> "Codes for special purposes" // XXII
		else -> "Unknown or Invalid"
	}
}

In [ ]:
val targetDirectory =
	"./COMP4037 - Research Methods/Coursework 2"

var df_hosp = DataFrame.readCsv(targetDirectory.plus("/NHS Hospital Admissions 2/").plus("Primary Diagnosis – 3 character code and description_cleaned.csv"), delimiter = ';')


In [ ]:
df_hosp = df_hosp.add("ICD10_Chapter") {
	mapIcd10Chapter(it.`Primary Diagnosis - 3 character code and description`)
}.moveTo(2, "ICD10_Chapter")

In [ ]:
val icd10_Chapters = df_hosp["ICD10_Chapter"].distinct().mapIndexed { i, chapter -> i+1 to chapter }.values
icd10_Chapters.forEach { println(it)}

In [ ]:
df_hosp = df_hosp.add("ICD10_Chapter_Code") {
	val chapter = it["ICD10_Chapter"] as String
	icd10_Chapters.filter { it.second == chapter }.first().first

}.moveTo(2, "ICD10_Chapter_Code")

In [ ]:
df_hosp

In [ ]:
val icd10_code_mapping = df_hosp.`Primary Diagnosis - 3 character code and description`.distinct().sort().mapIndexed { i, chapter -> i+1 to chapter }.values

icd10_code_mapping

In [ ]:
df_hosp = df_hosp.add("ICD10_Code_Mapping") {
	val code = it.`Primary Diagnosis - 3 character code and description`
	icd10_code_mapping.filter { it.second == code }.first().first

}.moveTo(4, "ICD10_Code_Mapping")

In [ ]:
df_hosp.`Primary Diagnosis - 3 character code and description`.distinct().sort().values

In [ ]:
val pop_file = DataFrame.readCsv(targetDirectory.plus("/").plus("ukpopulationestimates18382023_table3.csv"), delimiter = ';')

In [ ]:
pop_file

In [ ]:
pop_file.columnNames().zip(pop_file.columnTypes())

In [ ]:
var dfPopMelt = pop_file.gather { nameContains(Regex("[0-9]{4}")) }.into("Year", "Population").convert("Year").toInt()

In [ ]:
dfPopMelt

In [ ]:
dfPopMelt.columnNames().zip(dfPopMelt.columnTypes())

In [ ]:
val popTotal = dfPopMelt.filter { it["Age"] == "All Ages" && it["Sex"] == "Persons" }

In [ ]:
popTotal

In [ ]:
df_hosp.columnNames()

In [ ]:
df_hosp.columnNames().zip(df_hosp.columnTypes())

## calculate population growth and use it to analyze changes in diagnosis rates per capita for each gender/sex

In [ ]:
val popGender = dfPopMelt.filter { it["Age"] == "All Ages" && (it["Sex"] == "Males" || it["Sex"] == "Females") }
	.add("Gender") { if (it["Sex"] == "Males") "Male" else "Female" }
	.select("Year", "Gender", "Population")

val popGenderPrev = popGender
	.rename("Population" to "Prev_Population")
	.update { Year }.with { it + 1 }

var popGenderGrowthDf = popGender
	.innerJoin(popGenderPrev) { "Year" and "Gender" }


In [ ]:
popGenderGrowthDf = popGenderGrowthDf.add("Pop_YoY_Growth_Pct") {
	val currentPop = it.Population.toDouble()
	val prevPop = it.Prev_Population.toDouble()
	((currentPop - prevPop) / prevPop) * 100.0
}

In [ ]:
var hospByDiagGender = df_hosp.select { Year and "ICD10_Chapter" and "ICD10_Chapter_Code" and "ICD10_Code_Mapping" and `Primary Diagnosis - 3 character code and description` and Description and `Mean age` and `Finished Consultant Episodes` and Male and Female }

In [ ]:
hospByDiagGender.columnNames()

In [ ]:
hospByDiagGender = hospByDiagGender.update{ Female }.where { Female == 0 }.with {
	`Finished Consultant Episodes` - Male
}

In [ ]:
val hospGenderMelt = hospByDiagGender.gather { "Male" and "Female" }.into("Gender", "Admissions")

var diagGenderRatesCurrent = hospGenderMelt
	.innerJoin(popGender) { "Year" and "Gender" }

In [ ]:
diagGenderRatesCurrent

In [ ]:
diagGenderRatesCurrent = diagGenderRatesCurrent.add("Rate_Per_100k") {
	(it.Admissions.toDouble().div(it.Population.toDouble())).times(100000.0)
}
//	.select("Year", "Description", "Mean age", "Gender", "Admissions", "Rate_Per_100k")


In [ ]:
val diagGenderRatesPrev = diagGenderRatesCurrent
	.rename("Rate_Per_100k" to "Prev_Rate_Per_100k")
	.update { Year }.with { it + 1 }

var diagGenderGrowthDf = diagGenderRatesCurrent
	.innerJoin(diagGenderRatesPrev) { "Year" and "Description" and "Gender" }
	

In [ ]:
diagGenderGrowthDf = diagGenderGrowthDf.add("Rate_YoY_Growth_Pct") {
	val currentRate = it.Rate_Per_100k.toDouble()
	val prevRate = it.Prev_Rate_Per_100k.toDouble()
	if (prevRate > 0.0) {
		((currentRate - prevRate) / prevRate) * 100.0
	} else {
		0.0
	}
}

In [ ]:
var finalGenderAnalysisDf = diagGenderGrowthDf
	.innerJoin(popGenderGrowthDf.select("Year", "Gender", "Pop_YoY_Growth_Pct")) { "Year" and "Gender" }

In [ ]:
finalGenderAnalysisDf = finalGenderAnalysisDf.add("True_Surge_Index") {
	it.Rate_YoY_Growth_Pct.toDouble().minus(it.Pop_YoY_Growth_Pct.toDouble())
}
	.sortByDesc("True_Surge_Index")

In [ ]:
finalGenderAnalysisDf = finalGenderAnalysisDf.add("True_Surge_Index_SymLog") {
	val a = it["True_Surge_Index"] as Double
	ln1p(a.absoluteValue).times(a.sign)
}

In [ ]:
finalGenderAnalysisDf = finalGenderAnalysisDf.add("Gender_Code") {
	if (it.Gender == "Female") 1 else 0
}

In [ ]:
val newdf = finalGenderAnalysisDf

In [ ]:
val maxTrue_Surge_Index_SymLog =  newdf.True_Surge_Index_SymLog.max()
val minTrue_Surge_Index_SymLog =  newdf.True_Surge_Index_SymLog.min()

finalGenderAnalysisDf = newdf.add("True_Surge_Index_SymLogNormalized") {
	val a = it.True_Surge_Index_SymLog
	(a - minTrue_Surge_Index_SymLog)/(maxTrue_Surge_Index_SymLog - minTrue_Surge_Index_SymLog)
}


In [ ]:
val sortedDf = finalGenderAnalysisDf.sortBy("Description", "Gender", "Year")

var dfWithRollingAvg =
    sortedDf
        .filter { it.Year > 2012 }
        .groupBy("Description", "Gender")
        .updateGroups {

          val rates =
              this["Rate_YoY_Growth_Pct"].values().map { (it as? Number)?.toDouble() ?: 0.0 }

          val rollingMeans =
              rates.indices.map { i ->
                val start = max(0, i - 2)
                val window = rates.subList(start, i + 1)
                window.average()
              }

          add("Avg_YoY_Rate_Growth_3_Yr") { rollingMeans[index()] }
        }

In [ ]:
((739.8476 / 682.3543).pow(1.0/5.0).minus(1.0)).times(100.0)

In [ ]:
val dfCAGR = dfWithRollingAvg
		.updateGroups {
			val rates =
				this["Rate_Per_100k"].values().map { (it as? Number)?.toDouble() ?: 0.0 }

			val cagr =
				rates.indices.map { i ->
					val start = max(0, i - 4)
					val window = rates.subList(start, i + 1)
					val result = if (window.size == 5 && window.first() > 0.0 ) {
						val a = ((window.last() / window.first()).pow((1.0 / 5.0)).minus(1.0)).times(100)
						a
					} else 0.0
                    result
				}

			add("CAGR_%") { cagr[index()] }
		}
		.concat()

In [ ]:
dfCAGR

In [ ]:
//dfCAGR.filter { it.Year > 2012 }
//	.writeCsv("2013-2022_DiagnosisGenderCAGRAnalysis_Df.csv")

In [ ]:
dfCAGR.describe()


In [ ]:
finalGenderAnalysisDf.columnNames().forEach {
	if (finalGenderAnalysisDf[it].type.toString() == "kotlin.Double") {
		println(it)
	}
}

In [ ]:
//finalGenderAnalysisDf.writeCsv("DiagnosisGenderAnalysisDf.csv")